In [1]:
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchmetrics as tm

In [2]:
root_path = "/home/stefan/ioai-prep/kits/rezi"
device = "cuda" if torch.cuda.is_available() else "cpu"

seed = 42
torch.random.manual_seed(seed)

batch_size = 64

# Data

In [3]:
train_df = pd.read_csv(f"{root_path}/train.csv").sample(frac=0.1, random_state=seed)
test_df = pd.read_csv(f"{root_path}/test.csv")

train_df.head()

,SampleID,Question,Option0,Option1,Option2,Option3,Answer
79254,442d2f5a-f973-46a6-8dff-fff28574e35a,Which of the following is the source of hepati...,Limbus cells,ITO cells,Oval cells,Paneth cells,2
112920,db685621-a007-47bc-b117-db6ff71b4c06,Methotrexate is often used as a chemotherapeut...,G1 phase,S phase,G2 phase,M phase,1
114670,06df9638-8d07-4a4c-83b2-5742bfd05447,The given manifestation is the most common sym...,Insulinoma,Glucagonoma,Gastrinoma,Somatostatinoma,1
113521,dece6b72-b1c1-416c-989f-8393064555b7,"The WHO announced, immunisation is a primary r...",States,International community,Voluntary agencies,An individual,0
107795,a0506f15-281c-4e8b-ad67-1fe713fba743,Drug not given for Prophylaxis of Malaria is:,Chloroquine,Progaunil,Aeseunate,Doxycycline,2


In [4]:
class ReziDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512, is_train=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train

        mk_prompt = lambda row: (
            f"{row['Question']} [SEP] 0. {row['Option0']} [SEP] 1. {row['Option1']} "
            f"[SEP] 2. {row['Option2']} [SEP] 3. {row['Option3']}"
        )

        self.prompts = self.df.apply(mk_prompt, axis=1)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        enc = self.tokenizer(
            self.prompts[idx],
            add_special_tokens=True,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }

        if self.is_train:
            item["labels"] = torch.tensor(row["Answer"], dtype=torch.long)
        else:
            item["SampleID"] = row["SampleID"]
        return item

In [5]:
tokenizer = AutoTokenizer.from_pretrained("medicalai/ClinicalBERT")

train_full = ReziDataset(train_df, tokenizer)
train_sz = int(0.85 * len(train_full))
val_sz = len(train_full) - train_sz
train_ds, val_ds = torch.utils.data.random_split(train_full, [train_sz, val_sz])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

test_ds = ReziDataset(test_df, tokenizer, is_train=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

In [6]:
(train_full.prompts.apply(len) > 512).sum(), (test_ds.prompts.apply(len) > 512).sum()

(np.int64(170), np.int64(23))

In [7]:
# sanity check
x = next(iter(train_loader))
{k:x[k].shape for k in x.keys()}

{'input_ids': torch.Size([64, 512]),
 'attention_mask': torch.Size([64, 512]),
 'labels': torch.Size([64])}

# BERT setup

In [8]:
clinical_bert = AutoModel.from_pretrained("medicalai/ClinicalBERT").to(device)

In [9]:
sample_row = test_df.iloc[0]

sample_q = tokenizer(
    f"{sample_row['Question']} [SEP] 0. {sample_row['Option0']} [SEP] 1. {sample_row['Option1']} [SEP] 2. {sample_row['Option2']} [SEP] 3. {sample_row['Option3']}",
    add_special_tokens=True,
    padding="max_length",
    max_length=200,
    return_tensors="pt"
).to(device)

print(clinical_bert(**sample_q).last_hidden_state.shape)

cls_tok = clinical_bert(**sample_q).last_hidden_state[:, 0, :]

cls_tok.shape

torch.Size([1, 200, 768])


torch.Size([1, 768])

# Model

In [10]:
class MedicalBERT(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = clinical_bert
        for p in self.encoder.parameters():
            p.requires_grad = True

        self.head = nn.Sequential(
            nn.Linear(768, 256),
            nn.LeakyReLU(),
            nn.Linear(256, 32),
            nn.LeakyReLU(),
            nn.Linear(32, 4)
        )

    def forward(self, input_ids, attention_mask):
        logits = self.encoder(input_ids, attention_mask).last_hidden_state
        cls_tok = logits[:, 0, :]

        return self.head(cls_tok)
    
    def train(self, mode=True):
        super().train(mode)
        self.encoder.train(mode)
        return self

In [11]:
model = MedicalBERT().to(device)

model(x["input_ids"].to(device), x["attention_mask"].to(device)).shape

torch.Size([64, 4])

In [12]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

Trainable parameters: 134,939,300


# Training

In [13]:
epochs = 4
lr = 2e-5

optimizer = torch.optim.AdamW(model.parameters(), lr)
criterion = nn.CrossEntropyLoss()
accuracy = tm.Accuracy(task='multiclass', num_classes=4).to(device)

In [14]:
for epoch in range(1, epochs+1):
    model.train()
    running_loss, val_loss = 0.0, 0.0

    for sample in tqdm(train_loader):
        input_ids, attention_mask = sample["input_ids"].to(device), sample["attention_mask"].to(device)
        labels = sample["labels"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    model.eval()
    accuracy.reset()
    for sample in tqdm(val_loader):
        input_ids, attention_mask = sample["input_ids"].to(device), sample["attention_mask"].to(device)
        labels = sample["labels"].to(device)

        with torch.no_grad():
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            accuracy(logits, labels)

        val_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    val_loss = val_loss / len(val_loader)
    acc = accuracy.compute()
    print(f"epoch {epoch}, loss={avg_loss:.2f}, acc={acc*100:.1f}%, val_loss={val_loss:.2f}")

100%|██████████| 29/29 [00:07<00:00,  3.68it/s]


epoch 1, loss=1.36, acc=32.0%, val_loss=1.37


100%|██████████| 29/29 [00:07<00:00,  3.64it/s]


epoch 2, loss=1.35, acc=28.1%, val_loss=1.37


100%|██████████| 29/29 [00:07<00:00,  3.64it/s]


epoch 3, loss=1.34, acc=31.6%, val_loss=1.36


100%|██████████| 29/29 [00:08<00:00,  3.45it/s]

epoch 4, loss=1.29, acc=32.1%, val_loss=1.40


In [15]:
model.eval()
sub = []

with torch.no_grad():
    for sample in tqdm(test_loader):
        logits = model(sample["input_ids"].to(device),
                       sample["attention_mask"].to(device))
        preds = logits.argmax(dim=1).cpu().tolist()
        for sid, p in zip(sample["SampleID"], preds):
            sub.append((sid, p))

pd.DataFrame(sub, columns=["DatapointID", "PredictedAnswer"]) \
  .to_csv("submission_bert.csv", index=False)

100%|██████████| 44/44 [00:12<00:00,  3.41it/s]
